# GeoTIFF to COG

This notebook converts the clipped GeoTIFF files in `3d-map/data` into Cloud Optimized GeoTIFFs (COG).

It uses GDAL's `COG` driver and writes the outputs next to the source rasters.

In [5]:
from pathlib import Path
from osgeo import gdal

gdal.UseExceptions()

candidate_dirs = [
    Path.cwd() / '3d-map' / 'data',
    Path.cwd(),
]

for candidate in candidate_dirs:
    # if (candidate / 'dsm_clipped.tif').exists():
    if (candidate / 'slope_clipped.tif').exists():
        DATA_DIR = candidate.resolve()
        break
else:
    raise FileNotFoundError('Could not locate 3d-map/data with the expected source TIFF files.')

FILES = {
    # 'dsm_clipped.tif': 'dsm.tif',
    # 'orthophoto_rgb_clipped.tif': 'orthophoto_rgb.tif',
    # 'poa_clipped.tif': 'poa.tif',
    # 'poa_sweet_spots_clipped.tif': 'poa_sweet_spots.tif',
    'slope_clipped.tif': 'slope.tif',
    'aspect_clipped.tif': 'aspect.tif'
}

print(f'DATA_DIR = {DATA_DIR}')
for src_name, dst_name in FILES.items():
    print(f'{src_name} -> {dst_name}')

DATA_DIR = /Users/Alevtina/Documents/GitHub/sun-potential-3d-template/3d-map/data
slope_clipped.tif -> slope.tif
aspect_clipped.tif -> aspect.tif


In [6]:
def predictor_for_dataset(dataset):
    first_band = dataset.GetRasterBand(1)
    data_type_name = gdal.GetDataTypeName(first_band.DataType)
    if data_type_name.startswith('Float'):
        return '3'
    return '2'


def convert_geotiff_to_cog(src_path: Path, dst_path: Path) -> None:
    src_ds = gdal.Open(str(src_path), gdal.GA_ReadOnly)
    if src_ds is None:
        raise RuntimeError(f'Failed to open source raster: {src_path}')

    options = gdal.TranslateOptions(
        format='COG',
        creationOptions=[
            'COMPRESS=DEFLATE',
            f'PREDICTOR={predictor_for_dataset(src_ds)}',
            'BLOCKSIZE=512',
            'BIGTIFF=IF_SAFER',
            'NUM_THREADS=ALL_CPUS',
            'OVERVIEWS=IGNORE_EXISTING',
            'RESAMPLING=AVERAGE',
        ],
    )

    dst_ds = gdal.Translate(str(dst_path), src_ds, options=options)
    if dst_ds is None:
        raise RuntimeError(f'Failed to create COG: {dst_path}')

    dst_ds = None
    src_ds = None

    check_ds = gdal.Open(str(dst_path), gdal.GA_ReadOnly)
    image_structure = check_ds.GetMetadata('IMAGE_STRUCTURE')
    if image_structure.get('LAYOUT') != 'COG':
        raise RuntimeError(f'Output is missing COG layout metadata: {dst_path}')
    check_ds = None


for src_name, dst_name in FILES.items():
    src_path = DATA_DIR / src_name
    dst_path = DATA_DIR / dst_name

    if not src_path.exists():
        raise FileNotFoundError(f'Source file does not exist: {src_path}')

    print(f'Converting {src_path.name} -> {dst_path.name}')
    convert_geotiff_to_cog(src_path, dst_path)
    print(f'  Created {dst_path.name}')

print('Done.')

Converting slope_clipped.tif -> slope.tif
  Created slope.tif
Converting aspect_clipped.tif -> aspect.tif
  Created aspect.tif
Done.
